## Carregamento da base BETA

*Kalil Nege — RA 2409020 · OPT098 - Tópicos Especiais em Sistemas de Computação (Prof. Dr. Rodrigo Hübner)*

Na pasta `avi/` montei o roteiro do `SomeFeatures` com o AVI SSVEP Dataset, que tem um único canal. Aqui repito o mesmo caminho com o **BETA** (Liu et al., 2020, *Frontiers in Neuroscience*), que é bem maior:

- **70 participantes**, **64 canais** (sistema 10-10), gravados fora de sala blindada;
- 40 alvos num teclado virtual, de **8 a 15,8 Hz** com passo de 0,2 Hz, cada um com uma fase inicial diferente (JFPM);
- **4 blocos**, cada bloco com um ensaio por alvo (160 ensaios por participante);
- dados já filtrados de 3 a 100 Hz e reamostrados para **250 Hz** pelos autores;
- cada ensaio tem 0,5 s antes do início do estímulo e 0,5 s depois do fim. O estímulo dura **2 s em S1–S15** e **3 s em S16–S70**.

> Os arquivos `S1.mat` … `S70.mat` devem ficar em `dados/brutos/` (ao lado destes notebooks). O download é feito em http://bci.med.tsinghua.edu.cn/download.html.

In [1]:
from scipy.io import loadmat
import numpy as np

mat = loadmat('dados/brutos/S1.mat')
[k for k in mat if not k.startswith('__')]

['data']

In [2]:
dados = mat['data'][0, 0]
info = dados['suppl_info'][0, 0]
print('Campos:', dados.dtype.names)
print('Informações:', info.dtype.names)
print('\nFormato do EEG:', dados['EEG'].shape, '-> (canais, amostras, blocos, condições)')
print('Taxa de amostragem:', info['srate'][0, 0], 'Hz')

Campos: ('EEG', 'suppl_info')
Informações: ('sub', 'age', 'gender', 'chan', 'freqs', 'phases', 'bci_quotient', 'wide_snr', 'narrow_snr', 'srate')

Formato do EEG: (64, 750, 4, 40) -> (canais, amostras, blocos, condições)
Taxa de amostragem: 250 Hz


### Ordem das condições

A quarta dimensão do EEG é a condição (o alvo), mas ela **não segue a ordem das frequências**: a primeira condição é 8,6 Hz, e 8,0, 8,2 e 8,4 Hz ficam no final. Isso acontece porque a ordem é a do teclado (ponto, vírgula, apagar, letras…). Por isso uso sempre `suppl_info.freqs` e reordeno tudo por frequência.

In [3]:
freqs = info['freqs'].ravel()
print('Como vem no arquivo:', np.round(freqs[:6], 1), '...', np.round(freqs[-4:], 1))
print('Ordenado:           ', np.round(np.sort(freqs)[:6], 1), '...', np.round(np.sort(freqs)[-4:], 1))

Como vem no arquivo: [8.6 8.8 9.  9.2 9.4 9.6] ... [15.8  8.   8.2  8.4]
Ordenado:            [8.  8.2 8.4 8.6 8.8 9. ] ... [15.2 15.4 15.6 15.8]


### Duração dos ensaios

In [4]:
for s in (1, 16):
    eeg = loadmat(f'dados/brutos/S{s}.mat')['data'][0, 0]['EEG']
    print(f'S{s}: {eeg.shape[1]} amostras = {eeg.shape[1] / 250:.0f} s por ensaio')

S1: 750 amostras = 3 s por ensaio


S16: 1000 amostras = 4 s por ensaio


### Canais utilizados

Não preciso dos 64 canais. Separo dois grupos:

- **SSVEP**: os 9 canais parietais e occipitais que os próprios autores usaram na análise em tempo real (Pz, PO3, PO5, PO4, PO6, POz, O1, Oz, O2). É sobre o córtex visual que a resposta aparece com mais força.
- **Ritmos (beta/TBR)**: 9 canais frontais e centrais (F3, Fz, F4, FC1, FCz, FC2, C3, Cz, C4). O controle de atenção descendente (*top-down*) que a literatura associa ao beta e à razão teta/beta é medido em geral nessas regiões, e elas ficam mais longe do córtex visual, o que diminui (mas não zera) a mistura com o SSVEP.

In [5]:
CANAIS_SSVEP = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2']
CANAIS_RITMO = ['F3', 'FZ', 'F4', 'FC1', 'FCZ', 'FC2', 'C3', 'CZ', 'C4']
CANAIS = CANAIS_SSVEP + CANAIS_RITMO

nomes = [str(c[3][0]).strip().upper() for c in info['chan']]
print(len(nomes), 'canais no arquivo:', ' '.join(nomes))

64 canais no arquivo: FP1 FPZ FP2 AF3 AF4 F7 F5 F3 F1 FZ F2 F4 F6 F8 FT7 FC5 FC3 FC1 FCZ FC2 FC4 FC6 FT8 T7 C5 C3 C1 CZ C2 C4 C6 T8 M1 TP7 CP5 CP3 CP1 CPZ CP2 CP4 CP6 TP8 M2 P7 P5 P3 P1 PZ P2 P4 P6 P8 PO7 PO5 PO3 POZ PO4 PO6 PO8 CB1 O1 OZ O2 CB2


### Organização e salvamento

Para cada participante guardo só os 18 canais escolhidos, com as condições já em ordem de frequência, no formato `(blocos, alvos, canais, amostras)` e em `float32`. Também guardo as informações do participante, incluindo os SNRs médios e o quociente BCI calculados pelos autores (o quociente BCI é usado no notebook 05).

Primeiro testei com 5 participantes (S1, S2, S16, S17 e S18, misturando os dois formatos de ensaio) e depois rodei todos os 70.

In [6]:
import os

SUJEITOS = list(range(1, 71))
os.makedirs('dados/npz', exist_ok=True)


def carrega_beta(s):
    d = loadmat(f'dados/brutos/S{s}.mat')['data'][0, 0]
    info = d['suppl_info'][0, 0]
    nomes = [str(c[3][0]).strip().upper() for c in info['chan']]
    ordem = np.argsort(info['freqs'].ravel())
    idx = [nomes.index(c) for c in CANAIS]

    eeg = d['EEG'][idx][:, :, :, ordem]
    X = eeg.transpose(2, 3, 0, 1).astype(np.float32)
    extra = dict(freqs=np.round(info['freqs'].ravel()[ordem], 1),
                 idade=float(info['age'][0, 0]), genero=str(info['gender'][0]),
                 snr_estreita=float(info['narrow_snr'][0, 0]), snr_larga=float(info['wide_snr'][0, 0]),
                 quociente_bci=float(info['bci_quotient'][0, 0]))
    return X, extra


tabela = []
for s in SUJEITOS:
    X, extra = carrega_beta(s)
    np.savez(f'dados/npz/S{s}.npz', X=X, **extra)
    tabela.append((s, extra['idade'], extra['snr_estreita'], extra['snr_larga'], extra['quociente_bci']))

tabela = np.array(tabela)
np.savetxt('dados/participantes.csv', tabela, delimiter=',', fmt='%.4f',
           header='sujeito,idade,snr_estreita,snr_larga,quociente_bci', comments='')
print(len(SUJEITOS), 'participantes salvos. Formato do último:', X.shape)

70 participantes salvos. Formato do último: (4, 40, 18, 1000)


In [7]:
print(f'Idade: {tabela[:, 1].mean():.1f} ± {tabela[:, 1].std():.1f} anos')
print(f'SNR banda estreita (autores): {tabela[:, 2].mean():.2f} dB  |  banda larga: {tabela[:, 3].mean():.2f} dB')

Idade: 25.1 ± 7.9 anos
SNR banda estreita (autores): 4.00 dB  |  banda larga: -13.78 dB
